# Context-Aware Technical Manual Assistant (Mini RAG)

This notebook implements:
- Text preprocessing
- Chunking
- Keyword-based retrieval
- Question answering using Hugging Face

In [ ]:
import re
from transformers import pipeline

In [2]:
with open("manual.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

print(raw_text[:500])

INDUSTRIAL HYDRAULIC PRESS SYSTEM
OPERATION AND MAINTENANCE MANUAL

SECTION 1: INTRODUCTION
The industrial hydraulic press system is designed for high-pressure metal forming operations. It operates using hydraulic fluid to generate compressive force. The system includes a pressure valve, control panel, hydraulic pump, reservoir tank, and safety release mechanisms.

SECTION 2: SAFETY PRECAUTIONS
Before operating the machine, ensure all safety guards are in place. Operators must wear protective eq


In [3]:
def clean_text(text):
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

cleaned_text = clean_text(raw_text)

print(cleaned_text[:500])

INDUSTRIAL HYDRAULIC PRESS SYSTEM OPERATION AND MAINTENANCE MANUAL SECTION 1: INTRODUCTION The industrial hydraulic press system is designed for high-pressure metal forming operations. It operates using hydraulic fluid to generate compressive force. The system includes a pressure valve, control panel, hydraulic pump, reservoir tank, and safety release mechanisms. SECTION 2: SAFETY PRECAUTIONS Before operating the machine, ensure all safety guards are in place. Operators must wear protective equi


In [4]:
def chunk_text(text, chunk_size=200):
    words = text.split()   # simple split instead of nltk
    chunks = []
    
    for i in range(0, len(words), chunk_size):
        chunk = words[i:i+chunk_size]
        chunks.append(" ".join(chunk))

    print(len(chunks))
    print(chunks[0][:200])
    
    return chunks

In [5]:
chunks = chunk_text(cleaned_text)

3
INDUSTRIAL HYDRAULIC PRESS SYSTEM OPERATION AND MAINTENANCE MANUAL SECTION 1: INTRODUCTION The industrial hydraulic press system is designed for high-pressure metal forming operations. It operates usi


In [6]:
def keyword_score(chunk, query):
    chunk_words = set(chunk.lower().split())
    query_words = set(query.lower().split())
    return len(chunk_words.intersection(query_words))

def retrieve_top_chunks(chunks, query, top_k=2):
    scored = []
    
    for chunk in chunks:
        score = keyword_score(chunk, query)
        scored.append((chunk, score))
    
    scored = sorted(scored, key=lambda x: x[1], reverse=True)
    
    top_chunks = [chunk for chunk, score in scored[:top_k]]
    return top_chunks

In [7]:
query = "How do I reset the pressure valve?"

top_chunks = retrieve_top_chunks(chunks, query)

for i, ch in enumerate(top_chunks):
    print(f"\n--- Chunk {i+1} ---\n")
    print(ch[:500])


--- Chunk 1 ---

SECTION 5: NORMAL OPERATION During operation, monitor the pressure gauge continuously. Ensure that the pressure remains within the safe operating range. Adjust the pressure valve if necessary to maintain stable operation. Avoid sudden changes in pressure as it may damage internal components. SECTION 6: PRESSURE VALVE RESET PROCEDURE To reset the pressure valve, first turn off the machine using the main power switch. Allow the system pressure to drop to zero by releasing residual pressure through

--- Chunk 2 ---

INDUSTRIAL HYDRAULIC PRESS SYSTEM OPERATION AND MAINTENANCE MANUAL SECTION 1: INTRODUCTION The industrial hydraulic press system is designed for high-pressure metal forming operations. It operates using hydraulic fluid to generate compressive force. The system includes a pressure valve, control panel, hydraulic pump, reservoir tank, and safety release mechanisms. SECTION 2: SAFETY PRECAUTIONS Before operating the machine, ensure all safety guards are in place

In [ ]:
qa_pipeline = pipeline(
    "question-answering",
    model="distilbert-base-uncased-distilled-squad"
)

In [9]:
context = " ".join(top_chunks)

result = qa_pipeline(
    question=query,
    context=context
)

print("Answer:", result["answer"])
print("Confidence:", result["score"])

Answer: first turn off the machine using the main power switch
Confidence: 0.48114654421806335


In [10]:
if result["score"] < 0.2:
    print("Answer not found in manual.")
else:
    print("Final Answer:", result["answer"])

Final Answer: first turn off the machine using the main power switch
